In [ ]:
!pip install bs4
!pip install lxml
from urllib.request import urlopen
from bs4 import BeautifulSoup
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
with open('sample.html', 'r', encoding='utf-8') as file:
    html_content = file.read()

In [ ]:
# html tag 구조 시각화(이해X, 암기X)
from graphviz import Digraph
def visualize_html_structure(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    dot = Digraph(comment='HTML Structure', format='svg')  # SVG 형식으로 변경하여 더 좋은 텍스트 지원을 확보

    def add_nodes_and_edges(soup, parent_id=None):
        my_id = str(id(soup))
        if parent_id is not None:
            dot.edge(parent_id, my_id)

        if soup.name is not None:
            # 노드 라벨에 태그 이름과 텍스트 내용을 추가합니다.
            label = f'{soup.name}'
            if soup.text:
                # 텍스트 내용을 UTF-8로 인코딩
                text_summary = soup.text.strip()[:30] + '...' if len(soup.text.strip()) > 30 else soup.text.strip()
                text_summary = text_summary.encode('ascii', 'xmlcharrefreplace').decode('ascii')  # 특수 문자를 HTML 엔티티로 변환
                label += f'\n{text_summary}'
            dot.node(my_id, label)

        for child in soup.children:
            if child.name is not None:
                add_nodes_and_edges(child, my_id)

    add_nodes_and_edges(soup)
    return dot


dot = visualize_html_structure(html_content)
dot.render('html_structure', view=True)  # 그래프 파일을 생성하고 보여줍니다.

'html_structure.svg'

** 기초 파싱


In [ ]:
# BeautifulSoup 라이브러리를 사용하여 HTML 문서를 쉽게 분석(파싱)하는 코드

# html_content에 들어있는 HTML 문자열을 BeautifulSoup 객체로 변환
# 'html.parser'는 파이썬 기본 HTML 파서를 사용하겠다는 의미
soup = BeautifulSoup(html_content, 'html.parser')

# 파싱된 HTML 구조를 사람이 읽기 쉽게 정리된 형태로 출력
# prettify()는 HTML 태그를 들여쓰기 형태로 보기 좋게 만들어 줌
print(soup.prettify())

<!DOCTYPE html>
<html lang="ko">
 <head>
  <meta charset="utf-8"/>
  <title>
   덕성여자대학교 데이터사이언스학과
  </title>
  <style>
   /* 간단한 스타일 예시 */
        body {
            font-family: sans-serif;
            margin: 0;
            padding: 0;
        }
        header, nav, main, footer {
            padding: 10px 20px;
        }
        header {
            background-color: #f2f2f2;
        }
        nav {
            background-color: #e0e0e0;
        }
        nav ul {
            list-style-type: none;
            margin: 0;
            padding: 0;
        }
        nav li {
            display: inline-block;
            margin-right: 15px;
        }
        section {
            margin-bottom: 20px;
        }
        table {
            border-collapse: collapse;
            margin-bottom: 20px;
        }
        table, th, td {
            border: 1px solid #ccc;
            padding: 8px;
        }
        footer {
            background-color: #f2f2f2;
            text-align: center;

In [ ]:
# Parsing 연습하기 1 (지정된 한개의 태그만 파싱)

# soup.h1
# → HTML 문서에서 <h1> 태그 중 첫 번째 요소를 가져와 출력
print(soup.h1)
print("-------------")

# soup.div
# → <div> 태그 중 첫 번째 요소를 가져와 출력
# (여러 개가 있어도 첫 번째 것만 가져옴)
print(soup.div)
print("-------------")

# soup.ul
# → <ul> (unordered list, 순서 없는 목록) 태그 중 첫 번째 요소 출력
print(soup.ul)
print("-------------")

# soup.li
# → <li> (list item, 목록의 항목) 태그 중 첫 번째 요소 출력
# (여러 개 중 하나만 가져온다는 점이 중요)
print(soup.li)
print("-------------")

# soup.a
# → <a> (하이퍼링크) 태그 중 첫 번째 요소 출력
print(soup.a)
print("-------------")

# soup.section
# → <section> 태그 중 첫 번째 요소 출력
# (해당 태그가 없으면 None이 출력됨)
print(soup.section)

<h1>덕성여자대학교 데이터사이언스학과</h1>
-------------
<div class="form">
<label for="name">이름:</label>
<input id="name" name="name" required="" type="text"/>
</div>
-------------
<ul>
<li><a href="#intro">학과 소개</a></li>
<li><a href="#courses">교과목 안내</a></li>
<li><a href="#faculty">교수진 소개</a></li>
<li><a href="#resources">학생 지원</a></li>
<li><a href="#contact">문의하기</a></li>
</ul>
-------------
<li><a href="#intro">학과 소개</a></li>
-------------
<a href="#intro">학과 소개</a>
-------------
<section id="intro">
<h2>학과 소개</h2>
<p>
            데이터사이언스학과는 빅데이터 시대에 발맞춰 이론과 실무 능력을 겸비한
            인재를 양성하기 위해 개설되었습니다. 통계학, 컴퓨터공학, 머신러닝, 
            인공지능 등 다양한 분야의 융합 교육을 통해 폭넓은 진로를 모색할 수 있습니다.
        </p>
<img alt="학과 대표 이미지" src="https://via.placeholder.com/400x200"/>
</section>


In [ ]:
# Parsing 연습하기 2 (find() 메소드를 사용하여 특정 조건의 태그 1개만 가져오기)

# soup.find('h1')
# → <h1> 태그 중 첫 번째 요소를 찾아서 출력
# (여러 개 있어도 가장 처음 것만 가져옴)
print(soup.find('h1'))
print("---------------------------------------------")

print(soup.find('h2'))
print("---------------------------------------------")

# soup.find(id='resources')
# → id 속성이 "resources"인 태그를 찾아서 출력
# (id는 보통 문서 내에서 유일하므로 특정 요소를 정확히 찾을 때 사용)
print(soup.find(id='resources'))
print("---------------------------------------------")

# soup.find(id='faculty')
# → id가 "faculty"인 태그를 찾아 출력
print(soup.find(id='faculty'))
print("---------------------------------------------")

# soup.find(class_='form')
# → class 속성이 "form"인 태그를 찾아 출력
# (class는 여러 개 존재할 수 있지만, 첫 번째 것만 반환됨)
# ※ class는 파이썬 예약어라서 class_ 로 작성해야 함
print(soup.find(class_='form'))
print("---------------------------------------------")

# soup.find(attrs={'type': 'text'})
# → type="text" 속성을 가진 태그를 찾아 출력
# (예: <input type="text"> 같은 태그)
print(soup.find(attrs={'type': 'text'}))
print("---------------------------------------------")

# soup.find(attrs={'class': 'form'})
# → class="form" 속성을 가진 태그를 찾아 출력
# (class_='form'과 동일한 의미, 다른 표현 방식)
print(soup.find(attrs={'class': 'form'}))

<h1>덕성여자대학교 데이터사이언스학과</h1>
---------------------------------------------
<h2>학과 소개</h2>
---------------------------------------------
<section id="resources">
<h2>학생 지원</h2>
<p>
            데이터사이언스학과 학생들이 학업과 연구에 전념할 수 있도록 다양한 지원 
            프로그램과 시설을 제공합니다.
        </p>
<ul>
<li>전용 실습실 및 고성능 컴퓨팅 자원</li>
<li>학술 세미나 및 워크숍 개최</li>
<li>취업 및 진로 상담 프로그램 운영</li>
</ul>
</section>
---------------------------------------------
<section id="faculty">
<h2>교수진 소개</h2>
<ol>
<li>
<strong>홍길동 교수</strong> - 머신러닝, 빅데이터 전공
                <p>박사(통계학), 빅데이터 분석 프로젝트 다수 진행</p>
</li>
<li>
<strong>김영희 교수</strong> - 인공지능, 딥러닝 전공
                <p>박사(컴퓨터공학), 인공지능 연구실 운영</p>
</li>
<li>
<strong>박철수 교수</strong> - 데이터 시각화, 데이터 마이닝 전공
                <p>박사(정보과학), 시각화 솔루션 개발 경험 다수</p>
</li>
</ol>
</section>
---------------------------------------------
<div class="form">
<label for="name">이름:</label>
<input id="name" name="name" required="" type="text"/>
</div>
---------------------------------------------
<input i

In [ ]:
# Parsing 연습하기 3 (find_all() 메소드를 사용하여 조건에 맞는 태그를 모두 가져오기)

# soup.find_all('ul')
# → HTML 문서 안에 있는 모든 <ul> 태그를 리스트 형태로 가져옴
# (여러 개가 있으면 전부 포함됨)
print(soup.find_all('ul'))
print("---------------------------------------------")

# soup.find_all('li')
# → 모든 <li> 태그를 리스트로 가져옴
# (목록의 모든 항목을 한 번에 확인할 수 있음)
print(soup.find_all('li'))
print("---------------------------------------------")

# soup.find_all(class_='form')
# → class 속성이 "form"인 모든 태그를 리스트로 가져옴
# (조건에 맞는 여러 요소를 한 번에 수집할 때 사용)
print(soup.find_all(class_='form'))

[<ul>
<li><a href="#intro">학과 소개</a></li>
<li><a href="#courses">교과목 안내</a></li>
<li><a href="#faculty">교수진 소개</a></li>
<li><a href="#resources">학생 지원</a></li>
<li><a href="#contact">문의하기</a></li>
</ul>, <ul>
<li>통계학 입문</li>
<li>프로그래밍 기초(Python, R 등)</li>
<li>머신러닝 개론</li>
<li>데이터 시각화</li>
<li>빅데이터 분석</li>
<li>인공지능과 딥러닝</li>
</ul>, <ul>
<li>전용 실습실 및 고성능 컴퓨팅 자원</li>
<li>학술 세미나 및 워크숍 개최</li>
<li>취업 및 진로 상담 프로그램 운영</li>
</ul>]
---------------------------------------------
[<li><a href="#intro">학과 소개</a></li>, <li><a href="#courses">교과목 안내</a></li>, <li><a href="#faculty">교수진 소개</a></li>, <li><a href="#resources">학생 지원</a></li>, <li><a href="#contact">문의하기</a></li>, <li>통계학 입문</li>, <li>프로그래밍 기초(Python, R 등)</li>, <li>머신러닝 개론</li>, <li>데이터 시각화</li>, <li>빅데이터 분석</li>, <li>인공지능과 딥러닝</li>, <li>
<strong>홍길동 교수</strong> - 머신러닝, 빅데이터 전공
                <p>박사(통계학), 빅데이터 분석 프로젝트 다수 진행</p>
</li>, <li>
<strong>김영희 교수</strong> - 인공지능, 딥러닝 전공
                <p>박사(컴퓨터공학), 인공지능 연구실 운영</p>
</li>, <li>
<s

** element 파싱


In [ ]:
# 1. 제목 태그 <h1> ~ <h6>
for i in range(1, 7):
    for tag in soup.find_all(f"h{i}"):
        print(f"<h{i}>: {tag.text}")

<h1>: 덕성여자대학교 데이터사이언스학과
<h2>: 학과 소개
<h2>: 교과목 안내
<h2>: 교수진 소개
<h2>: 학생 지원
<h2>: 문의하기
<h3>: 개설 교과목 예시


In [ ]:
# 2. 단락 <p> 태그
print("\n단락 텍스트:")
for p in soup.find_all("p"):
    print("-", p.text.strip())


단락 텍스트:
- Department of Data Science, Dukseong Women's University
- 데이터사이언스학과는 빅데이터 시대에 발맞춰 이론과 실무 능력을 겸비한
            인재를 양성하기 위해 개설되었습니다. 통계학, 컴퓨터공학, 머신러닝, 
            인공지능 등 다양한 분야의 융합 교육을 통해 폭넓은 진로를 모색할 수 있습니다.
- 데이터사이언스학과에서 제공하는 주요 교과목은 아래와 같습니다.
- 박사(통계학), 빅데이터 분석 프로젝트 다수 진행
- 박사(컴퓨터공학), 인공지능 연구실 운영
- 박사(정보과학), 시각화 솔루션 개발 경험 다수
- 데이터사이언스학과 학생들이 학업과 연구에 전념할 수 있도록 다양한 지원 
            프로그램과 시설을 제공합니다.
- © 2025 Dukseong Women's University, Department of Data Science


In [ ]:
# 3. 하이퍼링크 <a>
print("\n하이퍼링크:")
for a in soup.find_all("a"):
    print(f"- 텍스트: {a.text.strip()}, 링크: {a.get('href')}")


하이퍼링크:
- 텍스트: 학과 소개, 링크: #intro
- 텍스트: 교과목 안내, 링크: #courses
- 텍스트: 교수진 소개, 링크: #faculty
- 텍스트: 학생 지원, 링크: #resources
- 텍스트: 문의하기, 링크: #contact


In [ ]:
# 4. 이미지 <img>
print("\n이미지 정보:")
for img in soup.find_all("img"):
    print(f"- src: {img.get('src')}, alt: {img.get('alt')}")


이미지 정보:
- src: https://via.placeholder.com/400x200, alt: 학과 대표 이미지


In [ ]:
# 5. 리스트 <ul> 및 <li>
print("\n리스트 항목:")
for ul in soup.find_all("ul"):
    items = ul.find_all("li")
    for li in items:
        print("-", li.text.strip())


리스트 항목:
- 학과 소개
- 교과목 안내
- 교수진 소개
- 학생 지원
- 문의하기
- 통계학 입문
- 프로그래밍 기초(Python, R 등)
- 머신러닝 개론
- 데이터 시각화
- 빅데이터 분석
- 인공지능과 딥러닝
- 전용 실습실 및 고성능 컴퓨팅 자원
- 학술 세미나 및 워크숍 개최
- 취업 및 진로 상담 프로그램 운영


In [ ]:
# 6. 리스트 <ol> 및 <li> (교수진 소개)
print("\n교수진 소개:")
for ol in soup.find_all("ol"):
    for li in ol.find_all("li"):
        print("-", li.text.strip())


교수진 소개:
- 홍길동 교수 - 머신러닝, 빅데이터 전공
                박사(통계학), 빅데이터 분석 프로젝트 다수 진행
- 김영희 교수 - 인공지능, 딥러닝 전공
                박사(컴퓨터공학), 인공지능 연구실 운영
- 박철수 교수 - 데이터 시각화, 데이터 마이닝 전공
                박사(정보과학), 시각화 솔루션 개발 경험 다수


In [ ]:
# 7. 테이블 데이터
print("\n개설 교과목 테이블:")
table = soup.find("table")
if table:
    headers = [th.text.strip() for th in table.find_all("th")]
    print(" | ".join(headers))
    for row in table.find_all("tr")[1:]:
        cells = [td.text.strip() for td in row.find_all("td")]
        print(" | ".join(cells))


개설 교과목 테이블:
과목명 | 학년/학기 | 학점 | 담당 교수
데이터베이스 개론 | 2학년 1학기 | 3 | 홍길동
고급 프로그래밍 | 2학년 2학기 | 3 | 김영희
머신러닝 프로젝트 | 3학년 2학기 | 3 | 박철수


In [ ]:
# 8. Form 입력 항목
print("\n문의하기 폼 필드:")
form = soup.find("form")
if form:
    inputs = form.find_all(["input", "textarea"])
    for field in inputs:
        print(f"- name: {field.get('name')}, type: {field.get('type', 'textarea')}")


문의하기 폼 필드:
- name: name, type: text
- name: email, type: email
- name: message, type: textarea


In [ ]:
# 9. <div> 태그 중 class='form' (form 그룹핑)
print("\n<form> 안의 <div class='form'> 내용:")
form_div = soup.find("div", class_="form")
if form_div:
    print(form_div.text.strip())


<form> 안의 <div class='form'> 내용:
이름:


Q1. 주어진 sample.html 에서 교수 이름만 출력하는 코드를 작성하시오.

In [ ]:
print("\n교수 이름:")
for professor in soup.find_all("strong"):
   print("-", professor.text.strip().replace('교수',''))


교수 이름:
- 홍길동 
- 김영희 
- 박철수 


In [ ]:
#교수님 답
# BeautifulSoup 객체에서 id가 "faculty"인 <section> 태그를 찾아 해당 섹션을 변수에 저장
faculty_section = soup.find("section", id="faculty")

# 해당 섹션에서 교수 이름이 들어 있는 <strong> 태그들을 모두 찾음 (보통 이름을 강조할 때 사용됨)
professors = faculty_section.find_all("strong")

# 교수진 목록 출력 제목을 표시
print("📘 교수진 이름 목록:")

# 찾은 각 <strong> 태그에서 텍스트(교수 이름)를 추출하여 출력
for prof in professors:
    # 텍스트를 가져와 공백을 제거하고, " 교수"라는 직함을 제거하여 이름만 남김
    name = prof.text.strip().replace(" 교수", "")

    # 이름 출력
    print("-", name)

📘 교수진 이름 목록:
- 홍길동
- 김영희
- 박철수
